# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/3bud-ZC/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My lane is **Lane 2 — Refresh / Content Opportunity Scoring**.

The operational task is **ranking / scoring**: given a limited editorial review budget, rank content pages so a strategist can inspect the most promising refresh or intervention candidates first. A binary classifier can be used underneath to estimate a decline-risk probability, but the product-facing output is not simply "decline / no decline" — it is an ordered **top-K review queue**.

This keeps the ML task tied to the real decision from Week 1: **which pages should be reviewed first?**

In [ ]:
import os, subprocess, json
from pathlib import Path
import pandas as pd

REPO_URL = "https://github.com/3bud-ZC/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data/raw/content_refresh_anonymized.csv").exists():
            return candidate
    return None

root = find_repo_root()
if root is None:
    if not Path(REPO_DIR).exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    root = Path(REPO_DIR).resolve()

os.chdir(root)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

with open("outputs/model_results.json", "r", encoding="utf-8") as fh:
    results = json.load(fh)

print("Task type: ranking / scoring")
print("Rows available:", len(df))
print("Decision output: top-K content review queue")


## 2. Target or proxy

For Week 2 I will use the starter repository's transparent **proxy label**:

`is_declining_proxy = 1` when `trend_direction == "down"`, otherwise 0.

This is useful for learning the workflow, but it is **not the final capstone target**. It is defined from the current measurement window, so it should be treated as a proxy rather than a future observed outcome. `trend_direction` and `trend_pct` must therefore never be used as model features when this proxy is the target.

For the later capstone, the stronger target will be an **observed future outcome** from the daily warehouse, such as whether a content item experiences a sustained decline in a future window after a clearly separated feature window. The exact future-window definition will be fixed in the data-contract assignment.

In [ ]:
df["is_declining_proxy"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("Proxy positives:", int(df["is_declining_proxy"].sum()))
print("Proxy positive rate:", round(df["is_declining_proxy"].mean(), 3))
print("Leakage columns excluded from features: trend_direction, trend_pct")

assert df["is_declining_proxy"].isin([0, 1]).all()


## 3. Success metric

The primary metric is **Precision@50** because the real decision has limited review capacity: if an editor can inspect roughly 50 pages, the quality of the top 50 matters more than overall accuracy.

I will call the ranking **good enough to continue** when:

- **Precision@50 ≥ 0.60** on a client-holdout evaluation, meaning at least 30 of the top 50 candidates are positive under the defined target; and
- it clearly beats the fixed-rule baseline on the same split and target.

This threshold is operational rather than cosmetic: 30 useful candidates in a 50-page review queue would be materially better than the verified starter fixed rule, which reached Precision@50 = 0.240. The current starter model result is reference evidence only; the future-target capstone must earn the result again.

In [ ]:
baseline_p50 = results["baseline"]["baseline_precision_at_50"]
rf_p50 = results["models"]["random_forest"]["precision_at_50"]
success_threshold = 0.60

print("Verified fixed-rule Precision@50:", round(baseline_p50, 3))
print("Verified random-forest Precision@50:", round(rf_p50, 3))
print("Provisional success threshold:", success_threshold)
print("Threshold met by current starter reference run:", rf_p50 >= success_threshold)
print("Top-50 positives at threshold:", int(success_threshold * 50))


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item at the current decision snapshot.**

The identifiers are used only to preserve row identity and client grouping; they are not model features. The feature columns shown below are observable search, engagement, freshness, and content signals available at the decision point. The final column is the Week 2 proxy target.

For the full warehouse later, this will evolve into **one content item at one decision date**, with features built strictly from the past and the target measured strictly in the future.

In [ ]:
lane_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "sessions_90d",
    "engagement_rate",
    "scroll_rate",
    "word_count",
    "is_declining_proxy",
]

lane_df = df[lane_columns].copy()

print("Lane dataframe shape:", lane_df.shape)
print("One row per content item:", lane_df["content_id"].is_unique)
print("Clients represented:", lane_df["client_id"].nunique())
lane_df.head(8)


## 5. Why ML beats a fixed rule here

A fixed rule remains a useful **baseline**, but the decision depends on many interacting signals: visibility, position, CTR, content age, freshness, engagement, depth, and client context. Their relationships can be nonlinear and can change across different parts of the data. A single if-statement has to hard-code those interactions and thresholds.

The verified starter run supports testing ML rather than stopping at the rule: under client-holdout validation, the fixed baseline reached **Precision@50 = 0.240**, while the random forest reached **0.680**. That result does not prove the model causes better content outcomes; it shows that a learned ranking can prioritize the current proxy more effectively than this fixed rule on the starter slice.

ML therefore earns a place here as a **ranking engine to beat a transparent rule**, while the rule remains the interpretability anchor and minimum benchmark.

In [ ]:
lift = rf_p50 / baseline_p50 if baseline_p50 else float("nan")
absolute_gain = rf_p50 - baseline_p50

print("Fixed-rule Precision@50:", round(baseline_p50, 3))
print("Learned-model Precision@50:", round(rf_p50, 3))
print("Absolute gain:", round(absolute_gain, 3))
print("Relative lift:", round(lift, 1), "x")
print("Decision claim: learned ranking improved prioritization of the defined proxy on this starter evaluation.")


## Self-check

Before submission:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries are published
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] The ML task type, target/proxy, success metric, unit of analysis, and fixed-rule comparison are explicit
- [ ] Committed to my repo under `work/notebooks/`
- [ ] Submit the public repository URL on the ML-03 card